# Multi-label Tweet Topic Classification

This notebook demonstrates a multi-label tweet topic classification task using a dataset from Hugging Face. The goal is to preprocess tweet data, build a deep learning model (LSTM-based), train it, and evaluate its performance on classifying tweets into various topics.

In [113]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("cardiffnlp/tweet_topic_multi")

In [114]:
ds

DatasetDict({
    test_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 573
    })
    test_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 1679
    })
    train_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 4585
    })
    train_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 1505
    })
    train_all: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 6090
    })
    validation_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 573
    })
    validation_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 188
    })
    train_random: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 4564
    })
    validati

In [115]:
type(ds)

datasets.dataset_dict.DatasetDict

In [116]:
import pandas as pd
train = pd.DataFrame(ds['train_random'])
display(train.head())

test = pd.DataFrame(ds['test_2021'])
valid = pd.DataFrame(ds['validation_random'])

,text,date,label,label_name,id
0,The latest The Movie theater Daily! {{URL}} Th...,2021-03-07,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...",[film_tv_&_video],1368464923370676231
1,Why is the internet dropping out every few min...,2020-02-16,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[science_&_technology],1229072651634737162
2,The reserve team man of the match sponsored by...,2021-06-13,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[sports],1403984572547690497
3,Coming to the end of #InternationalWomenDay202...,2020-03-08,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",[news_&_social_concern],1236768682422022144
4,tfw when you remember Selina Meyer was a ficti...,2020-11-08,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...",[film_tv_&_video],1325256551825862657


In [117]:
# Explode the list into individual rows and count
label_counts = train['label_name'].explode().value_counts()

print(label_counts)

label_name
news_&_social_concern       1515
sports                      1208
music                        774
film_tv_&_video              663
celebrity_&_pop_culture      641
diaries_&_daily_life         624
arts_&_culture               235
business_&_entrepreneurs     229
other_hobbies                220
science_&_technology         216
fitness_&_health             204
gaming                       176
relationships                171
family                       149
food_&_dining                117
learning_&_educational       111
fashion_&_style              105
travel_&_adventure            87
youth_&_student_life          64
Name: count, dtype: int64


In [118]:
# Define the labels you want to completely remove
labels_to_remove = [
    'learning_&_educational',
    'fashion_&_style',
    'travel_&_adventure',
    'youth_&_student_life'
]

# Create a mask: keep rows where NONE of the target labels appear
mask = ~train['label_name'].apply(lambda x: any(label in x for label in labels_to_remove))

# Apply the mask
train = train[mask].copy()

print(f"Original size: {len(train) + sum(~mask)}")
print(f"New size after removal: {len(train)}")

Original size: 4564
New size after removal: 4236


In [119]:
mask = ~valid['label_name'].apply(lambda x: any(label in x for label in labels_to_remove))

# Apply the mask
valid = valid[mask].copy()

print(f"Original size: {len(valid) + sum(~mask)}")
print(f"New size after removal: {len(valid)}")

Original size: 573
New size after removal: 538


In [120]:
mask = ~test['label_name'].apply(lambda x: any(label in x for label in labels_to_remove))

# Apply the mask
test = test[mask].copy()

print(f"Original size: {len(test) + sum(~mask)}")
print(f"New size after removal: {len(test)}")

Original size: 1679
New size after removal: 1597


In [121]:
# These are the original 19 minus the 4 you removed
kept_classes = [
    'arts_&_culture',
    'business_&_entrepreneurs',
    'celebrity_&_pop_culture',
    'diaries_&_daily_life',
    'family',
    'film_tv_&_video',
    'fitness_&_health',
    'food_&_dining',
    'gaming',
    'music',
    'news_&_social_concern',
    'other_hobbies',
    'relationships',
    'science_&_technology',
    'sports'
]

In [122]:
from sklearn.preprocessing import MultiLabelBinarizer

# Initialize the binarizer with your kept classes
mlb = MultiLabelBinarizer(classes=kept_classes)

# Transform the label_name column (list of lists) into a 2D array
# Shape: (n_samples, 15)
y_encoded = mlb.fit_transform(train['label_name'])

# Replace the old 'label' column with the new encoding
train['label'] = y_encoded.tolist()

# Update num_classes
num_classes = len(kept_classes)  # Should be 15

print(f"Shape of new label array: {y_encoded.shape}")
print(f"New num_classes: {num_classes}")
print(f"Classes: {mlb.classes_}")

Shape of new label array: (4236, 15)
New num_classes: 15
Classes: ['arts_&_culture' 'business_&_entrepreneurs' 'celebrity_&_pop_culture'
 'diaries_&_daily_life' 'family' 'film_tv_&_video' 'fitness_&_health'
 'food_&_dining' 'gaming' 'music' 'news_&_social_concern' 'other_hobbies'
 'relationships' 'science_&_technology' 'sports']


In [123]:
from sklearn.preprocessing import MultiLabelBinarizer

# Initialize the binarizer with your kept classes
mlb = MultiLabelBinarizer(classes=kept_classes)

# Transform the label_name column (list of lists) into a 2D array
# Shape: (n_samples, 15)
y_encoded = mlb.fit_transform(valid['label_name'])

# Replace the old 'label' column with the new encoding
valid['label'] = y_encoded.tolist()

# Update num_classes
num_classes = len(kept_classes)  # Should be 15

print(f"Shape of new label array: {y_encoded.shape}")
print(f"New num_classes: {num_classes}")
print(f"Classes: {mlb.classes_}")

Shape of new label array: (538, 15)
New num_classes: 15
Classes: ['arts_&_culture' 'business_&_entrepreneurs' 'celebrity_&_pop_culture'
 'diaries_&_daily_life' 'family' 'film_tv_&_video' 'fitness_&_health'
 'food_&_dining' 'gaming' 'music' 'news_&_social_concern' 'other_hobbies'
 'relationships' 'science_&_technology' 'sports']


In [124]:
from sklearn.preprocessing import MultiLabelBinarizer

# Initialize the binarizer with your kept classes
mlb = MultiLabelBinarizer(classes=kept_classes)

# Transform the label_name column (list of lists) into a 2D array
# Shape: (n_samples, 15)
y_encoded = mlb.fit_transform(test['label_name'])

# Replace the old 'label' column with the new encoding
test['label'] = y_encoded.tolist()

# Update num_classes
num_classes = len(kept_classes)  # Should be 15

print(f"Shape of new label array: {y_encoded.shape}")
print(f"New num_classes: {num_classes}")
print(f"Classes: {mlb.classes_}")

Shape of new label array: (1597, 15)
New num_classes: 15
Classes: ['arts_&_culture' 'business_&_entrepreneurs' 'celebrity_&_pop_culture'
 'diaries_&_daily_life' 'family' 'film_tv_&_video' 'fitness_&_health'
 'food_&_dining' 'gaming' 'music' 'news_&_social_concern' 'other_hobbies'
 'relationships' 'science_&_technology' 'sports']


In [125]:
def remove_cols(df):
  return df.drop(columns=['date','id'])

def labels_array_to_string(df):
  for i in range(0,df.shape[0]):
    if len(df.loc[i,'label_name']) > 0:
      df.loc[i,'label_name'] = df['label_name'][i][0]
    else :
      df.loc[i,'label_name'] = ''
  return df


In [126]:
!pip install urlextract

In [127]:
import re
from urlextract import URLExtract
extractor = URLExtract()

def format_tweet(tweet):
    # mask web urls
    urls = extractor.find_urls(tweet)
    for url in urls:
        tweet = tweet.replace(url, "{{URL}}")
    # format twitter account
    tweet = re.sub(r"\b(\s*)(@[\S]+)\b", r'\1{\2@}', tweet)
    return tweet

#Data Preprocessing

## Data Preprocessing Overview

This section handles the initial preparation of the raw tweet data. It involves loading the datasets into pandas DataFrames, dropping irrelevant columns like `date` and `id`, and transforming the `label_name` column from an array of labels to a single string representation. Additionally, the `format_tweet` function is applied to clean and standardize tweet text by masking URLs and formatting Twitter account mentions.

In [128]:
train = remove_cols(train)
test = remove_cols(test)
valid = remove_cols(valid)


In [129]:
# train = labels_array_to_string(train)
# test = labels_array_to_string(test)
# valid = labels_array_to_string(valid)

In [130]:
train['text'] = train['text'].apply(format_tweet)
test['text'] = test['text'].apply(format_tweet)
valid['text'] = valid['text'].apply(format_tweet)

In [131]:
train = train[train['label_name'] != '' ]

In [132]:
train.isnull().sum()

,0
text,0
label,0
label_name,0


In [133]:
test.isnull().sum()

,0
text,0
label,0
label_name,0


In [134]:
valid.isnull().sum()

,0
text,0
label,0
label_name,0


## Tokenization

In this step, the `text` data from the tweets is tokenized. Tokenization involves breaking down the raw text into individual words or subwords (tokens). A custom tokenizer is used to extract relevant terms, converting them to lowercase and handling hashtags and mentions. The resulting tokens are stored in a new `tokens` column in the DataFrame.

In [135]:
import re
import nltk
nltk.download('punkt')
from nltk.tokenize import TweetTokenizer

# Create the tokenizer with preserve_case=True (this keeps "CoD" and "FPS" intact)
tknzr = TweetTokenizer(preserve_case=False, strip_handles=False, reduce_len=True)

def tokenizer(text):
    return tknzr.tokenize(text)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [136]:
train['tokens'] = train['text'].apply(tokenizer)
test['tokens'] = test['text'].apply(tokenizer)
valid['tokens'] = valid['text'].apply(tokenizer)

## Vocabulary Mapping

This section creates a numerical representation for each unique word (token) in the dataset. A vocabulary map (`vocab`) is built where each unique token is assigned a unique integer ID. Special tokens like `<PAD>` and `<UNK>` (unknown) are also included. The `tokens` are then converted into numerical sequences (`sequence`) based on this vocabulary. This step also determines the total vocabulary size (`vocab_size`) and the maximum sequence length (`max_len`) observed in the training data.

In [137]:
# def vocab_map(df):
#   vocab = {
#       '<PAD>': 0,
#       '<UNK>': 1
#   }

#   idx = 2

#   for tokens in df["tokens"]:
#       for word in tokens:
#           if word not in vocab:
#               vocab[word] = idx
#               idx += 1

#   df["sequence"] = df["tokens"].apply(
#       lambda tokens: [vocab.get(word, 1) for word in tokens]
#   )
#   return df, len(vocab), len(df['tokens'].max())

# train, vocab_size, max_len= vocab_map(train)

In [138]:
# 1. Build vocab ONLY on training tokens
def build_vocab(df):
    vocab = {'<PAD>': 0, '<UNK>': 1}
    idx = 2
    for tokens in df["tokens"]:
        for word in tokens:
            if word not in vocab:
                vocab[word] = idx
                idx += 1
    return vocab

train_vocab = build_vocab(train)

# 2. Encode all splits using the SAME vocab
def encode_sequences(df, vocab):
    df["sequence"] = df["tokens"].apply(
        lambda tokens: [vocab.get(word, 1) for word in tokens]
    )
    return df

train = encode_sequences(train, train_vocab)
test  = encode_sequences(test, train_vocab)
valid = encode_sequences(valid, train_vocab)

vocab_size = len(train_vocab)
max_len    = max(train["tokens"].apply(len))   # compute only from training

In [139]:
train.head()

,text,label,label_name,tokens,sequence
0,The latest The Movie theater Daily! {{URL}} Th...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]",[film_tv_&_video],"[the, latest, the, movie, theater, daily, !, {...","[2, 3, 2, 4, 5, 6, 7, 8, 8, 9, 10, 10, 11, 12,..."
1,Why is the internet dropping out every few min...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]",[science_&_technology],"[why, is, the, internet, dropping, out, every,...","[16, 17, 2, 18, 19, 20, 21, 22, 23, 24, 25, 26..."
2,The reserve team man of the match sponsored by...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]",[sports],"[the, reserve, team, man, of, the, match, spon...","[2, 34, 35, 36, 37, 2, 38, 39, 40, 8, 41, 42, ..."
3,Coming to the end of #InternationalWomenDay202...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]",[news_&_social_concern],"[coming, to, the, end, of, #internationalwomen...","[63, 12, 2, 64, 37, 65, 26, 2, 66, 67, 68, 69,..."
4,tfw when you remember Selina Meyer was a ficti...,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]",[film_tv_&_video],"[tfw, when, you, remember, selina, meyer, was,...","[82, 83, 84, 85, 86, 87, 47, 56, 88, 89, 8, 90..."


## Padding

To ensure that all input sequences to the neural network have a uniform length, padding is applied. Sequences shorter than `max_len` are padded with `0` (corresponding to `<PAD>` token), while sequences longer than `max_len` are truncated. This creates a `padded_sequence` column, which is essential for batch processing in deep learning models.

In [140]:
def padding(df):
    padded_dataset = []

    for sequence in df['sequence']:
        if len(sequence) < max_len:
            sequence = sequence + [0] * (max_len - len(sequence))
        else:
            sequence = sequence[:max_len]

        padded_dataset.append(sequence)

    df['padded_sequence'] = padded_dataset
    return df


In [141]:
train = padding(train)
test  = padding(test)
valid = padding(valid)

# Embedding with LSTM and Classifier


##GloVe

In [142]:
def build_fasttext_matrix(vocab, ft_model, embedding_dim=300):
    embedding_matrix = np.zeros((len(vocab), embedding_dim))

    for word, idx in vocab.items():
        try:
            # FastText handles OOV words automatically!
            embedding_matrix[idx] = ft_model.get_word_vector(word)
        except:
            # If it fails, keep zero vector
            pass

    return embedding_matrix

In [143]:
import numpy as np
def load_glove(vocab):
    embedding_matrix = np.zeros((len(vocab), 100))
    with open('glove.6B.100d.txt', 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            # If the lowercase version exists in your vocab, use it
            if word in vocab:
                idx = vocab[word]
                embedding_matrix[idx] = np.array(values[1:], dtype='float32')
            # Also try without # and @ if you want (optional)
    return embedding_matrix

In [144]:
# # !pip install fasttext
# import fasttext.util

# # Download English vectors (will save ~3.6GB)
# fasttext.util.download_model('en', if_exists='ignore')
# ft = fasttext.load_model('cc.en.100.bin')

In [145]:
!wget https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip
!unzip glove.6B.zip

--2026-06-30 10:47:23--  https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip
Resolving huggingface.co (huggingface.co)... 3.165.160.59, 3.165.160.61, 3.165.160.12, ...
Connecting to huggingface.co (huggingface.co)|3.165.160.59|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/621ffdc136468d709f18095d/ded22944ac944002f423d1cdb83de044851330e1a7fe19d74dc59ffa632ad294?X-Xet-Cas-Uid=public&response-content-type=application%2Fzip&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27glove.6B.zip%3B+filename%3D%22glove.6B.zip%22%3B&user_id=public&Expires=1782820043&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjIxZmZkYzEzNjQ2OGQ3MDlmMTgwOTVkL2RlZDIyOTQ0YWM5NDQwMDJmNDIzZDFjZGI4M2RlMDQ0ODUxMzMwZTFhN2ZlMTlkNzRkYzU5ZmZhNjMyYWQyOTRcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LXR5cGU9YXBwbGljYXRpb24lMkZ6aXAmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj1pbmxpb

In [146]:
embedding_matrix = load_glove(train_vocab)

## Model Building

Here, a sequential deep learning model is constructed using Keras. The model architecture consists of:

1.  **Embedding Layer**: Converts integer-encoded words into dense vectors of fixed size.
2.  **LSTM Layer**: A Long Short-Term Memory recurrent layer, suitable for processing sequential data like text, with dropout for regularization.
3.  **Dense Layers**: Fully connected layers with ReLU activation for feature transformation.
4.  **Dropout Layer**: Further regularization to prevent overfitting.
5.  **Output Dense Layer**: A final dense layer with `sigmoid` activation, appropriate for multi-label classification, where each output neuron corresponds to a class and predicts the probability of that class being present.

The model is compiled with the `adam` optimizer and `binary_crossentropy` loss function, which is standard for multi-label tasks.

In [147]:
train['padded_sequence'].apply(len).unique()

array([118])

In [148]:
import numpy as np

X_train = np.array(train['padded_sequence'].tolist())

print(X_train.shape)
print(X_train.dtype)
type(train['label'][0])
y_train = np.array(train['label'].tolist())
print(y_train.shape)

(4236, 118)
int64
(4236, 15)


In [149]:
# Convert padded sequences to numpy arrays for test and validation
X_test = np.array(test['padded_sequence'].tolist())
X_valid = np.array(valid['padded_sequence'].tolist())

# Convert labels to numpy arrays for test and valid
y_test = np.array(test['label'].tolist())
y_valid = np.array(valid['label'].tolist())

print('Shape of X_test:', X_test.shape)
print('Shape of y_test:', y_test.shape)
print('Shape of X_valid:', X_valid.shape)
print('Shape of y_valid:', y_valid.shape)

Shape of X_test: (1597, 118)
Shape of y_test: (1597, 15)
Shape of X_valid: (538, 118)
Shape of y_valid: (538, 15)


In [150]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np


y_train = np.array(train['label'].tolist())


num_classes = y_train.shape[1]
print(f"✅ num_classes set to: {num_classes}")


✅ num_classes set to: 15


In [151]:
class_weights = {}
for i in range(num_classes):
    class_weights[i] = compute_class_weight(
        class_weight='balanced',
        classes=np.array([0, 1]),
        y=y_train[:, i]
    )[1]  # weight for class 1 (positive)

print("✅ Class weights computed successfully!")

✅ Class weights computed successfully!


In [152]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

def build_model(vocab_size, max_len, num_classes):
    model = Sequential([
        Embedding(vocab_size, 100, weights=[embedding_matrix], input_length=max_len, trainable=True),
        Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='sigmoid')   # multi-label
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',                # correct for multi-label
        metrics=['binary_accuracy']                # better than 'accuracy'
    )
    return model

In [153]:
model = build_model(vocab_size, max_len, num_classes)
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │     1,977,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,977,600 (7.54 MB)

 Trainable params: 1,977,600 (7.54 MB)

 Non-trainable params: 0 (0.00 B)

Forming testing and validation data

## Training and Testing the Model

This section involves training the defined LSTM model and evaluating its performance. Class weights are computed to address potential class imbalance in the training data, ensuring that the model doesn't overemphasize majority classes. The model is then trained using the `X_train` and `y_train` data, with `X_valid` and `y_valid` used for validation during training. After training, the model's performance is assessed on the unseen `X_test` and `y_test` data, and a classification report provides detailed metrics such as precision, recall, and F1-score for each class.

In [159]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=30,
    batch_size=64,
    class_weight=class_weights,   # compute per-class weights for multi-label
    callbacks=[early_stop]
)

Epoch 1/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 52s 759ms/step - binary_accuracy: 0.8952 - loss: 1.5618 - val_binary_accuracy: 0.9017 - val_loss: 0.2769
Epoch 2/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 81s 760ms/step - binary_accuracy: 0.9001 - loss: 1.4713 - val_binary_accuracy: 0.9141 - val_loss: 0.2522
Epoch 3/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 51s 761ms/step - binary_accuracy: 0.9118 - loss: 1.3546 - val_binary_accuracy: 0.9244 - val_loss: 0.2342
Epoch 4/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 86s 823ms/step - binary_accuracy: 0.9163 - loss: 1.2707 - val_binary_accuracy: 0.9257 - val_loss: 0.2210
Epoch 5/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 50s 748ms/step - binary_accuracy: 0.9201 - loss: 1.1853 - val_binary_accuracy: 0.9283 - val_loss: 0.2165
Epoch 6/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 50s 744ms/step - binary_accuracy: 0.9232 - loss: 1.1352 - val_binary_accuracy: 0.9279 - val_loss: 0.2110
Epoch 7/30
67/67 ━━━━━━━━━━━━━━━━━━━━ 51s 755ms/step - binary_accuracy: 0.9253 - loss: 1.0730 - val_binary_accuracy: 0.9296 - val_loss: 0.2082

In [160]:
y_pred_proba = model.predict(X_test)
y_pred_binary = (y_pred_proba > 0.5).astype(int)

from sklearn.metrics import f1_score
print("Weighted F1:", f1_score(y_test, y_pred_binary, average='weighted'))

50/50 ━━━━━━━━━━━━━━━━━━━━ 7s 148ms/step
Weighted F1: 0.5644160209255709


In [161]:
y_pred_proba = model.predict(X_test)
y_pred_binary = (y_pred_proba > 0.5).astype(int)
from sklearn.metrics import f1_score
print(f1_score(y_test, y_pred_binary, average='weighted'))

50/50 ━━━━━━━━━━━━━━━━━━━━ 8s 161ms/step
0.5644160209255709


In [162]:
from sklearn.metrics import classification_report

y_pred_proba = model.predict(X_test)
y_pred_binary = (y_pred_proba > 0.5).astype(int)

# Get class names
# class_names = train['label_name'].unique()
print(classification_report(y_test, y_pred_binary, target_names=kept_classes))

# Also check which classes are NEVER predicted
print("\nClasses never predicted:")
for i, name in enumerate(kept_classes):
    if y_pred_binary[:, i].sum() == 0:
        print(f"  - {i},{name}")

50/50 ━━━━━━━━━━━━━━━━━━━━ 8s 160ms/step
                          precision    recall  f1-score   support

          arts_&_culture       0.33      0.08      0.12        39
business_&_entrepreneurs       0.58      0.48      0.53        66
 celebrity_&_pop_culture       0.44      0.09      0.15       230
    diaries_&_daily_life       0.75      0.16      0.26       131
                  family       1.00      0.24      0.39        29
         film_tv_&_video       0.75      0.27      0.40       288
        fitness_&_health       0.71      0.44      0.55        54
           food_&_dining       0.43      0.40      0.41        15
                  gaming       0.69      0.35      0.47        77
                   music       0.94      0.77      0.84       367
   news_&_social_concern       0.71      0.62      0.66       307
           other_hobbies       0.00      0.00      0.00        79
           relationships       0.85      0.20      0.32        56
    science_&_technology       1.0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [164]:
import numpy as np
from sklearn.metrics import f1_score
y_pred_proba = model.predict(X_valid)
# For each class, find best threshold
best_thresholds = {}
for i in range(num_classes):
    best_f1 = 0
    best_thresh = 0.5
    for thresh in np.arange(0.1, 0.9, 0.05):
        y_pred_binary = (y_pred_proba[:, i] > thresh).astype(int)
        f1 = f1_score(y_valid[:, i], y_pred_binary)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    best_thresholds[i] = best_thresh
    print(f"Class {kept_classes[i]}: best threshold = {best_thresh:.2f} (F1={best_f1:.3f})")

# Apply class-specific thresholds
y_pred_binary_custom = np.zeros_like(y_pred_proba)
for i in range(num_classes):
    y_pred_binary_custom[:, i] = (y_pred_proba[:, i] > best_thresholds[i]).astype(int)

print("\nCustom threshold Weighted F1:",
      f1_score(y_valid, y_pred_binary_custom, average='weighted'))

17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 208ms/step
Class arts_&_culture: best threshold = 0.20 (F1=0.271)
Class business_&_entrepreneurs: best threshold = 0.40 (F1=0.538)
Class celebrity_&_pop_culture: best threshold = 0.15 (F1=0.492)
Class diaries_&_daily_life: best threshold = 0.10 (F1=0.463)
Class family: best threshold = 0.35 (F1=0.545)
Class film_tv_&_video: best threshold = 0.30 (F1=0.547)
Class fitness_&_health: best threshold = 0.20 (F1=0.339)
Class food_&_dining: best threshold = 0.15 (F1=0.686)
Class gaming: best threshold = 0.35 (F1=0.419)
Class music: best threshold = 0.30 (F1=0.860)
Class news_&_social_concern: best threshold = 0.55 (F1=0.836)
Class other_hobbies: best threshold = 0.10 (F1=0.214)
Class relationships: best threshold = 0.20 (F1=0.412)
Class science_&_technology: best threshold = 0.15 (F1=0.320)
Class sports: best threshold = 0.30 (F1=0.904)

Custom threshold Weighted F1: 0.6583378622733116


In [165]:
model.save('tweet_v1.keras')

In [168]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, hamming_loss

# 1. Get probabilities on the test set
y_test_proba = model.predict(X_test)

# 2. Apply the thresholds you found on validation
y_test_binary_custom = np.zeros_like(y_test_proba)
for i in range(num_classes):
    y_test_binary_custom[:, i] = (y_test_proba[:, i] > best_thresholds[i]).astype(int)

# 3. Calculate Weighted F1 (your main metric)
final_f1 = f1_score(y_test, y_test_binary_custom, average='weighted')
print(f"Final Test Weighted F1 (with custom thresholds): {final_f1:.4f}")

50/50 ━━━━━━━━━━━━━━━━━━━━ 10s 196ms/step
Final Test Weighted F1 (with custom thresholds): 0.6431


In [169]:
# Exact Match (strict accuracy)
exact_match = accuracy_score(y_test, y_test_binary_custom)

# Hamming Accuracy (per-label accuracy)
hamming_acc = 1 - hamming_loss(y_test, y_test_binary_custom)

# Binary Accuracy (Keras style)
binary_acc = (y_test == y_test_binary_custom).mean()

print("\n--- Test Set Performance (with Custom Thresholds) ---")
print(f"Exact Match Accuracy (strict):   {exact_match:.4f}")
print(f"Hamming Accuracy (per-label):    {hamming_acc:.4f}")
print(f"Binary Accuracy (Keras style):   {binary_acc:.4f}")
print(f"Weighted F1 (primary metric):    {final_f1:.4f}")


--- Test Set Performance (with Custom Thresholds) ---
Exact Match Accuracy (strict):   0.2862
Hamming Accuracy (per-label):    0.9001
Binary Accuracy (Keras style):   0.9001
Weighted F1 (primary metric):    0.6431


In [170]:
from sklearn.metrics import classification_report

print("\n--- Classification Report (Test Set, Custom Thresholds) ---")
print(classification_report(
    y_test,
    y_test_binary_custom,
    target_names=kept_classes,
    zero_division=0,
    digits=3
))

# Optional: check which classes are still never predicted
print("\nClasses never predicted on test set:")
for i, name in enumerate(kept_classes):
    if y_test_binary_custom[:, i].sum() == 0:
        print(f"  - {name}")


--- Classification Report (Test Set, Custom Thresholds) ---
                          precision    recall  f1-score   support

          arts_&_culture      0.054     0.333     0.093        39
business_&_entrepreneurs      0.528     0.576     0.551        66
 celebrity_&_pop_culture      0.282     0.761     0.411       230
    diaries_&_daily_life      0.260     0.771     0.388       131
                  family      0.500     0.276     0.356        29
         film_tv_&_video      0.549     0.608     0.577       288
        fitness_&_health      0.418     0.519     0.463        54
           food_&_dining      0.231     0.600     0.333        15
                  gaming      0.423     0.390     0.405        77
                   music      0.882     0.834     0.857       367
   news_&_social_concern      0.727     0.590     0.651       307
           other_hobbies      0.135     0.342     0.194        79
           relationships      0.580     0.518     0.547        56
    science_&_

In [172]:
import pickle
from tensorflow.keras.models import save_model

# 1. Save the model
save_model(model, 'tweet_v1.keras')
print("✅ Model saved as 'tweet_v1.keras'")

# 2. Save the vocabulary
with open('train_vocab.pkl', 'wb') as f:
    pickle.dump(train_vocab, f)
print("✅ Vocabulary saved as 'train_vocab.pkl'")

# 3. Save the configuration (max_len, num_classes)
config = {
    'max_len': max_len,
    'num_classes': num_classes
}
with open('config.pkl', 'wb') as f:
    pickle.dump(config, f)
print("✅ Config saved as 'config.pkl'")

# 4. Save the thresholds (your best thresholds from validation)
with open('best_thresholds.pkl', 'wb') as f:
    pickle.dump(best_thresholds, f)
print("✅ Thresholds saved as 'best_thresholds.pkl'")

# 5. Save the class names
with open('kept_classes.pkl', 'wb') as f:
    pickle.dump(kept_classes, f)
print("✅ Class names saved as 'kept_classes.pkl'")

✅ Model saved as 'tweet_v1.keras'
✅ Vocabulary saved as 'train_vocab.pkl'
✅ Config saved as 'config.pkl'
✅ Thresholds saved as 'best_thresholds.pkl'
✅ Class names saved as 'kept_classes.pkl'


#Apply

In [174]:
import pickle
import numpy as np
import re
from tensorflow.keras.models import load_model
from nltk.tokenize import TweetTokenizer
from urlextract import URLExtract

# --- Load all artifacts ---
def load_artifacts():
    model = load_model('tweet_v1.keras')
    with open('train_vocab.pkl', 'rb') as f:
        vocab = pickle.load(f)
    with open('config.pkl', 'rb') as f:
        config = pickle.load(f)
    with open('best_thresholds.pkl', 'rb') as f:
        thresholds = pickle.load(f)
    with open('kept_classes.pkl', 'rb') as f:
        class_names = pickle.load(f)

    return model, vocab, config, thresholds, class_names

# --- Preprocessing functions (must match training EXACTLY) ---
extractor = URLExtract()
tknzr = TweetTokenizer(preserve_case=False, strip_handles=False, reduce_len=True)  # Must match your training tokenizer

def format_tweet(tweet):
    urls = extractor.find_urls(tweet)
    for url in urls:
        tweet = tweet.replace(url, "{{URL}}")
    tweet = re.sub(r"\b(\s*)(@[\S]+)\b", r'\1{\2@}', tweet)
    return tweet

def tokenizer(text):
    return tknzr.tokenize(text)

# --- Main Prediction Function ---
def predict_tweet(text, model, vocab, config, thresholds, class_names):
    max_len = config['max_len']

    # 1. Preprocess
    cleaned = format_tweet(text)
    tokens = tokenizer(cleaned)

    # 2. Encode to integers (use 1 for unknown words)
    sequence = [vocab.get(word, 1) for word in tokens]  # 1 = <UNK>

    # 3. Pad/truncate to max_len
    if len(sequence) < max_len:
        sequence = sequence + [0] * (max_len - len(sequence))
    else:
        sequence = sequence[:max_len]

    # 4. Predict probabilities
    proba = model.predict(np.array([sequence]), verbose=0)[0]

    # 5. Apply thresholds (convert to binary labels)
    threshold_list = [thresholds[i] for i in range(len(class_names))]
    predictions = (proba > threshold_list).astype(int)

    # 6. Get final class names
    predicted_classes = [class_names[i] for i, val in enumerate(predictions) if val == 1]

    return predicted_classes, proba

# --- Test it! ---
if __name__ == "__main__":
    model, vocab, config, thresholds, class_names = load_artifacts()

    test_tweet = "I just bought a new PS5 and I'm playing Call of Duty all night!"
    classes, probs = predict_tweet(test_tweet, model, vocab, config, thresholds, class_names)

    print(f"Tweet: {test_tweet}")
    print(f"Predicted Classes: {classes}")

    # Print probabilities for top 3
    sorted_idx = np.argsort(probs)[::-1]
    print("\nTop 3 Probabilities:")
    for i in sorted_idx[:3]:
        print(f"  {class_names[i]}: {probs[i]:.4f}")

Tweet: I just bought a new PS5 and I'm playing Call of Duty all night!
Predicted Classes: ['diaries_&_daily_life', 'gaming']

Top 3 Probabilities:
  gaming: 0.4635
  film_tv_&_video: 0.1677
  diaries_&_daily_life: 0.1187
